# 02. 행정동별 고령층 커뮤니케이션 부족도

## 구현 기본안
5개 변수(통화대상자 수, 문자대상자 수, 통화량, 문자량, SNS)를 각각 20% 반영한다.
성별 코드는 원자료 1·2를 그대로 사용하므로 남녀 코드명을 추정해 연결하지 않는다.
연령코드 60·65·70·75를 사용한다. 공식 안내의 75세 이상 설명과 파일의 최상위 코드에 따라
75를 75세 이상으로 해석하는 구현 가정을 사용한다. 제공기관의 세부 코드 정의 변경 시 재검증한다.
10세 단위 격자 인구를 5세 단위로 임의 분할하지 않는다.

$$L_{dgk}=1-\frac{r_{dgk}-1}{m_{gk}-1}$$
$$C_{dg}=\frac{1}{5}\sum_{k=1}^5 L_{dgk},\quad
w_g=\frac{\sum_d M_{dg}}{\sum_{g,d}M_{dg}},\quad C_d=\sum_g w_g C_{dg}$$
$M$은 같은 통신파일의 총인구수이다. 표본수로 해석하지 않는다.
동을 격자에 복제하기 **전** 원본 424개 동에서 순위와 가중치를 계산한다.

## 자료 의미와 제한
평균·분위수는 매뉴얼상 원칙적으로 0이 아닌 관측을 대상으로 집계한다. 무조건부 개인 평균으로 해석하지 않는다.
SNS는 원래 Z-score이므로 음수를 유지하고 다른 변수와 함께 순위로 변환한다.
마스킹·결측은 0으로 채우지 않고 검증 실패로 처리한다.
이 점수는 취약계층 개인의 고립확률이나 100m 실측 통신점수가 아니다.

출처: [서울 시민생활 데이터](https://data.seoul.go.kr/dataVisual/seoul/seoulLiving.do),
[29개 통신정보](https://data.seoul.go.kr/dataList/OA-22267/F/1/datasetView.do),
사용자 제공 2022년 매뉴얼 6·14·24·35쪽. 기준 파일은 2025년 12월이며 최근 3개월 집계 성격을 가진다.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

start = Path.cwd().resolve()
PROJECT_ROOT = next(p for p in [start, *start.parents]
                    if (p / 'data/raw').is_dir() and (p / 'code').is_dir())
OUT = PROJECT_ROOT / 'data/processed/nonphysical_index'
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 160)

def rank01(values):
    # 결측을 정상 점수로 바꾸지 않는다. 비교대상이 하나면 순위를 정의할 수 없다.
    assert values.notna().all() and np.isfinite(values).all()
    assert len(values) > 1, '순위 비교대상이 2개 미만입니다.'
    return (values.rank(method='average') - 1) / (len(values) - 1)

def save_checked(frame, name, keys):
    path = OUT / name
    assert not frame.duplicated(keys).any()
    frame.to_csv(path, index=False, encoding='utf-8-sig')
    saved = pd.read_csv(path, encoding='utf-8-sig')
    assert len(saved) == len(frame) and not saved.duplicated(keys).any()
    for col in frame.select_dtypes(include='number'):
        assert np.allclose(frame[col], saved[col], equal_nan=True), col
    print('저장 및 재검증:', path.relative_to(PROJECT_ROOT))
    return saved

## 1. 통신자료와 성·연령 키 확인

In [2]:
source = PROJECT_ROOT / 'data/raw/population/demographics/senior/source/2025.12월_29개 통신정보.xlsx'
raw = pd.read_excel(source)
features = ['평균 통화대상자 수','평균 문자대상자 수','평균 통화량','평균 문자량','SNS 사용횟수']
id_cols = ['행정동코드','자치구','행정동','성별','연령대']
raw['연령대'] = pd.to_numeric(raw['연령대'], errors='raise')
assert set(raw['연령대']) == set(range(20,80,5)), '원본 연령구간 변경 확인 필요'
telecom = raw.loc[raw['연령대'].isin([60,65,70,75]), id_cols + ['총인구수'] + features].copy()
assert not telecom.duplicated(['행정동코드','성별','연령대']).any()
for col in ['총인구수'] + features:
    telecom[col] = pd.to_numeric(telecom[col], errors='raise')
assert telecom.notna().all().all()
assert np.isfinite(telecom[['총인구수'] + features]).all().all()
assert telecom['총인구수'].gt(0).all()
assert telecom[features[:-1]].ge(0).all().all()
assert set(telecom['성별']) == {1,2}
assert telecom.groupby('행정동코드').size().eq(8).all()
assert telecom.groupby('행정동코드')[['자치구','행정동']].nunique().eq(1).all().all()
assert not telecom[['행정동코드','자치구','행정동']].drop_duplicates().duplicated(['자치구','행정동']).any()
display(telecom.groupby(['성별','연령대']).agg(행정동수=('행정동코드','nunique'), 인구=('총인구수','sum')))

행정동수         인구
성별 연령대                 
1  60    424  360971.00
   65    424  299896.00
   70    424  218125.64
   75    424  305795.36
2  60    424  390519.00
   65    424  339850.00
   70    424  253863.53
   75    424  392277.47

## 2. 변수별 역순위와 공통 성·연령 가중치

In [3]:
groups = ['성별','연령대']
score_cols = []
for feature in features:
    col = feature + '_부족도'
    telecom[col] = 1 - telecom.groupby(groups)[feature].transform(rank01)
    score_cols.append(col)
telecom['성연령_부족도'] = telecom[score_cols].mean(axis=1)
weights = telecom.groupby(groups, as_index=False)['총인구수'].sum()
weights['공통가중치'] = weights['총인구수'] / weights['총인구수'].sum()
assert np.isclose(weights['공통가중치'].sum(), 1)
telecom = telecom.merge(weights[groups + ['공통가중치']], on=groups, validate='many_to_one')
telecom['가중부족도'] = telecom['성연령_부족도'] * telecom['공통가중치']
dong = telecom.groupby(['행정동코드','자치구','행정동'], as_index=False).agg(
    커뮤니케이션부족도=('가중부족도','sum'), 통신60세이상인구=('총인구수','sum'))
assert dong['커뮤니케이션부족도'].between(0,1).all()
display(weights)
display(dong['커뮤니케이션부족도'].describe())

,성별,연령대,총인구수,공통가중치
0,1,60,360971.00,0.140933
1,1,65,299896.00,0.117088
2,1,70,218125.64,0.085162
3,1,75,305795.36,0.119391
4,2,60,390519.00,0.152469
5,2,65,339850.00,0.132687
6,2,70,253863.53,0.099115
7,2,75,392277.47,0.153156


count    424.000000
mean       0.500000
std        0.132725
min        0.085579
25%        0.403651
50%        0.511629
75%        0.594341
max        0.789558
Name: 커뮤니케이션부족도, dtype: float64

## 3. 변수 중복과 SNS 제외 민감도 확인

In [4]:
# 성·연령 내 표준화된 변수끼리 상관을 확인한다.
display(telecom[score_cols].corr(method='spearman').round(3))
telecom['SNS제외_가중부족도'] = telecom[score_cols[:-1]].mean(axis=1) * telecom['공통가중치']
without_sns = telecom.groupby('행정동코드')['SNS제외_가중부족도'].sum()
comparison = dong.set_index('행정동코드')[['커뮤니케이션부족도']].join(without_sns)
print('SNS 제외 전후 행정동 순위 상관:', comparison.corr(method='spearman').iloc[0,1])
# 점수에 동률이 있는 경우 상위 10% 경계의 동을 모두 포함한다.
sets = [set(comparison.index[comparison[c] >= comparison[c].quantile(.9)]) for c in comparison]
print('상위 10% 집합 Jaccard:', len(sets[0] & sets[1]) / len(sets[0] | sets[1]))

,평균 통화대상자 수_부족도,평균 문자대상자 수_부족도,평균 통화량_부족도,평균 문자량_부족도,SNS 사용횟수_부족도
평균 통화대상자 수_부족도,1.000,0.717,0.607,0.048,0.391
평균 문자대상자 수_부족도,0.717,1.000,0.538,0.130,0.199
평균 통화량_부족도,0.607,0.538,1.000,0.006,0.296
평균 문자량_부족도,0.048,0.130,0.006,1.000,-0.003
SNS 사용횟수_부족도,0.391,0.199,0.296,-0.003,1.000


SNS 제외 전후 행정동 순위 상관: 0.9774227217984746
상위 10% 집합 Jaccard: 0.72


## 4. 산정 결과 저장

In [5]:
save_checked(dong, 'dong_communication_score.csv', ['행정동코드'])
save_checked(telecom, 'dong_communication_components.csv', ['행정동코드','성별','연령대'])

저장 및 재검증: data\processed\nonphysical_index\dong_communication_score.csv
저장 및 재검증: data\processed\nonphysical_index\dong_communication_components.csv


,행정동코드,자치구,행정동,성별,연령대,총인구수,평균 통화대상자 수,평균 문자대상자 수,평균 통화량,평균 문자량,SNS 사용횟수,평균 통화대상자 수_부족도,평균 문자대상자 수_부족도,평균 통화량_부족도,평균 문자량_부족도,SNS 사용횟수_부족도,성연령_부족도,공통가중치,가중부족도,SNS제외_가중부족도
0,1101053,종로구,사직동,1,60,396.0,15.63,5.16,50.65,8.71,-0.06,0.787234,0.247045,0.640662,0.962175,0.977541,0.722931,0.140933,0.101885,0.092914
1,1101053,종로구,사직동,1,65,326.0,13.92,4.65,50.21,9.49,-0.03,0.840426,0.208038,0.191489,0.905437,0.304965,0.490071,0.117088,0.057381,0.062800
2,1101053,종로구,사직동,1,70,198.0,13.78,4.69,39.96,13.15,-0.06,0.170213,0.030733,0.333333,0.371158,0.120567,0.205201,0.085162,0.017475,0.019277
3,1101053,종로구,사직동,1,75,340.0,9.34,3.53,31.54,12.75,-0.20,0.206856,0.096927,0.178487,0.320331,0.323877,0.225296,0.119391,0.026898,0.023956
4,1101053,종로구,사직동,2,60,374.0,14.54,5.03,64.11,13.48,-0.14,0.073286,0.108747,0.912530,0.354610,0.990544,0.487943,0.152469,0.074396,0.055239
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3387,1125074,강동구,길동,1,75,1444.0,8.45,2.81,28.66,11.59,-0.19,0.442080,0.555556,0.411348,0.439716,0.206856,0.411111,0.119391,0.049083,0.055179
3388,1125074,강동구,길동,2,60,2061.0,12.92,4.07,72.88,14.66,-0.09,0.504728,0.788416,0.221040,0.217494,0.795508,0.505437,0.152469,0.077064,0.066007
3389,1125074,강동구,길동,2,65,1852.0,12.88,3.77,78.73,11.15,-0.07,0.193853,0.516548,0.018913,0.552009,0.186761,0.293617,0.132687,0.038959,0.042504
3390,1125074,강동구,길동,2,70,1328.0,10.85,3.13,67.71,10.40,-0.14,0.427896,0.601655,0.269504,0.433806,0.540189,0.454610,0.099115,0.045059,0.042938
